In [4]:
import glob
import os
import pandas as pd

In [ ]:
base_path = 'shadow/examples/docs/tor-mal-exit/shadow.data'
data = []

# Match all files in the shadow.data directory
files = glob.glob(os.path.join(base_path, 'hosts/**/tor.*.stdout'), recursive=True)
    
for file in files:
    # This split gets 'exit_malicious'
    host_name = file.split(os.sep)[-2]
    
    with open(file, 'r', errors='ignore') as f:
        for line in f:
            data.append({'host': host_name, 'message': line.strip()})

df = pd.DataFrame(data)

In [6]:
# Count successes grouped by host
host_success_counts = df[df['message'].str.contains("FFS-ZKP handshake successful", na=False)] \
                        .groupby('host')['message'].count()

print("Handshake Successes per Exist Node")
print(host_success_counts)

Handshake Successes per Exist Node
host
exit2    2782
exit3    1124
exit4    1439
Name: message, dtype: int64


In [ ]:
# Malicious Attacks (Attack 1)
malicious_host = "exit1"

caught_attempts = df[df['host'] == malicious_host]['message'].str.contains(r"MaliciousExit", na=False).sum()

successful_handshakes = df[df['host'] == malicious_host]['message'].str.contains(r"FFS-ZKP handshake successful", na=False).sum()

total_attempts = caught_attempts + successful_handshakes

print(f"Total attempts through Malicious {malicious_host}: {total_attempts}")
print(f"Malicious attempts caught & blocked: {caught_attempts}")
print(f"Successful handshakes: {successful_handshakes}\n")

if successful_handshakes == 0:
    print("SUCCESS: Zero false negatives for Malicious Exit")
else:
    print(f"WARNING: {successful_handshakes} attempts were not successfully rejected")

Total attempts through Malicious exit1: 4655
Malicious attempts caught & blocked: 4655
Successful handshakes: 0

SUCCESS: Zero false negatives for Malicious Exit
